# SRD Honest Benchmark — Colab Runner

Runs the SRD perplexity sweep on TinyLlama-1.1B and produces the data behind `docs/SRD_RESULTS.md`.

**Pipeline:** install → unit tests → coherence smoke test → SRD perplexity sweep → K-quant baseline → plot → download.

**Hardware:** Colab T4 (free tier). Total wall-clock ~30-45 min including model download. Most cells are fast; the SRD sweep (Cell 5) is the long pole at ~10-15 min.

**Deliverables this notebook produces (downloaded in Cell 8):**
- `srd_sweep.json` — rows 1-5 of the results table (FP16 + 4 SRD configs)
- `kquant_sweep.json` — rows 6-9 (K-quant baselines, cited)
- `srd_perplexity_vs_bpw.png` — the headline plot

Paste the two JSON files back into the chat and the write-up at `docs/SRD_RESULTS.md` gets filled in.

**Source:** github.com/Orivael-Dev/axiom · plan at `/root/.claude/plans/abundant-wandering-salamander.md`

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Sweep will run on CPU (~3 hours instead of ~15 min).")
    print("Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Clone the AXIOM repo + install research/quant requirements
#
# GIT_REF points at the SRD prototype branch. Change to 'main' once
# the PR is merged.
import os
GIT_REF = "claude/srd-prototype-benchmark-JRtv1"

!rm -rf axiom
!git clone --depth 1 --branch {GIT_REF} https://github.com/Orivael-Dev/axiom.git
os.chdir("axiom")
!pip install -q -r research/quant/requirements.txt
print("\n[setup] ready. branch:", GIT_REF)


In [ ]:
# Cell 3: Phase A — unit tests (no model download, ~2 sec)
#
# Confirms the kernel reproduces the same numbers the unit tests pin:
#   - bpw matches hand calc (4 + 8 + 32/G + 32/G)
#   - residue strictly improves MSE
#   - shape / dtype / value-range invariants hold
# If any of these fail, halt — the sweep below is meaningless until
# the kernel is right.
import os
os.environ.setdefault("AXIOM_MASTER_KEY", "colab_smoke_key_only")
!python -m pytest tests/test_axiom_quant.py -q

In [ ]:
# Cell 4: Phase B — coherence smoke test
#
# Downloads TinyLlama-1.1B (~2 GB cached after first run), applies
# SRD at alpha=1, generates 80 tokens. The output should be coherent
# English. If the model spits gibberish at alpha=1, the kernel is
# bug-bait — STOP HERE and re-check axiom_quant.py.
#
# At alpha=0 the output should be noticeably degraded but still
# English-shaped (4-bit only is lossier than the published
# Q4_K_M numbers because there's no per-block scale tuning).
!python research/quant/quantize_model.py \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --alpha 1.0 \
    --prompt "Once upon a time, in a small village, " \
    --tokens 60

In [ ]:
# Cell 5: Phase C1 — SRD perplexity sweep (rows 1-5)
#
# Configs run: FP16 baseline, SRD alpha={0, 0.5, 1.0} per-block g=64,
# SRD alpha=1.0 per-tensor. Each config gets fresh weights so the
# previous run can't contaminate the next. Sliding-window PPL with
# stride=512, context=2048 (standard llama.cpp conventions).
#
# Sanity warning fires if the FP16 baseline drifts >0.5 from the
# published TinyLlama PPL (~7.7) — that means the eval harness is
# broken, not the kernel.
#
# Wall-clock: ~10-15 min on T4. The dataset fingerprint guard
# writes results/wikitext_fingerprint.txt on first run.
!python -m research.quant.bench_perplexity \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --group-size 64 \
    --output research/quant/results/srd_sweep.json

In [ ]:
# Cell 6: Phase C2 — K-quant baseline (rows 6-9)
#
# Cites the published llama.cpp PPLs for TinyLlama-1.1B at
# Q4_K_M / Q5_K_M / Q6_K / Q8_0. Fast (~1 sec, no download).
# Stride convention may differ from our SRD eval — see write-up.
#
# For stride-matched apples-to-apples, run Cell 6A then Cell 6B
# instead of this cell. They build llama.cpp from source (cmake)
# and run llama-perplexity --ppl-stride 512. ~30 min on H100.

!python -m research.quant.bench_llamacpp \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --output research/quant/results/kquant_sweep.json
print("Done — source=published_cite (stride caveat applies)")


In [ ]:
# Cell 6A: Install llama-cpp-python with CUDA
# Pre-built wheels (~30 sec). Falls back to source build if no wheel matches.
# Run once per Colab session, then run Cell 6B.
import subprocess, sys, os, re

# Detect CUDA version
r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
m = re.search(r'release (\d+)\.(\d+)', r.stdout + r.stderr)
if m:
    cuda_tag = f"cu{int(m.group(1))}{int(m.group(2)):02d}"
    print(f'CUDA detected: {m.group(1)}.{m.group(2)} → {cuda_tag}')
else:
    cuda_tag = 'cu121'
    print(f'CUDA not detected, trying {cuda_tag}')

wheel_url = f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_tag}'
print(f'Trying pre-built wheel from {wheel_url} ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'llama-cpp-python', '--extra-index-url', wheel_url],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Pre-built wheel unavailable, building from source with CUDA (~5 min)...')
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'llama-cpp-python', '--no-cache-dir', '-q'],
        check=True
    )
else:
    print('Pre-built wheel installed.')

from llama_cpp import Llama
print('llama-cpp-python OK')


In [ ]:
# Cell 6B: K-quant PPL via llama-cpp-python — no binary needed
# Requires Cell 6A. Downloads TheBloke GGUFs, computes sliding-window PPL
# with stride=512/context=2048 matching bench_perplexity.py exactly.
# Saves after each quant — safe to re-run if a quant fails.
import math, json, time, numpy as np
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

GGUF_REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'
GGUF_DIR  = Path('/tmp/srd_gguf')
GGUF_DIR.mkdir(exist_ok=True)
OUTPUT    = Path('research/quant/results/kquant_sweep.json')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
CONTEXT, STRIDE = 2048, 512

QUANTS = {
    'Q4_K_M': {'bpw': 4.85, 'file': 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf'},
    'Q5_K_M': {'bpw': 5.69, 'file': 'tinyllama-1.1b-chat-v1.0.Q5_K_M.gguf'},
    'Q6_K':   {'bpw': 6.56, 'file': 'tinyllama-1.1b-chat-v1.0.Q6_K.gguf'},
    'Q8_0':   {'bpw': 8.50, 'file': 'tinyllama-1.1b-chat-v1.0.Q8_0.gguf'},
}

# ── WikiText-2 ───────────────────────────────────────────────────────
wt2 = Path('/tmp/wt2.txt')
if not wt2.exists():
    print('Writing WikiText-2 test split...')
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    wt2.write_text('\n\n'.join(ds['text']))
wt2_bytes = wt2.read_bytes()
print(f'WikiText-2: {len(wt2_bytes) // 1024} KB')

# ── Perplexity kernel ────────────────────────────────────────────────
def _log_softmax(x):
    x = x.astype(np.float32)
    x -= x.max(axis=-1, keepdims=True)
    np.exp(x, out=x)
    x /= x.sum(axis=-1, keepdims=True)
    return np.log(x + 1e-30)

def compute_ppl(gguf_path):
    llm = Llama(model_path=str(gguf_path), n_ctx=CONTEXT,
                n_gpu_layers=-1, logits_all=True, verbose=False)
    tokens = llm.tokenize(wt2_bytes, add_bos=True)
    n_tokens = len(tokens)
    print(f'  Tokens: {n_tokens}')

    nlls, prev_end = [], 0
    windows = list(range(0, n_tokens - 1, STRIDE))

    for w, begin in enumerate(windows):
        end   = min(begin + CONTEXT, n_tokens)
        chunk = tokens[begin:end]
        if len(chunk) < 2:
            break

        llm.reset()
        llm.eval(chunk)

        # scores[i] = logits predicting chunk[i+1]
        logits = np.array(llm.scores[:len(chunk)], dtype=np.float32)
        lp     = _log_softmax(logits)

        first = max(0, prev_end - begin)   # skip already-counted context
        for i in range(first, len(chunk) - 1):
            nlls.append(-lp[i, chunk[i + 1]])

        prev_end = end
        if (w + 1) % 50 == 0 or end == n_tokens:
            print(f'  window {w+1}/{len(windows)}  '
                  f'running PPL={math.exp(float(np.mean(nlls))):.4f}')
        if end == n_tokens:
            break

    return math.exp(float(np.mean(nlls))), len(nlls)

# ── Resume from partial results ──────────────────────────────────────
rows = []
if OUTPUT.exists():
    rows = [r for r in json.loads(OUTPUT.read_text())
            if r.get('source') == 'rerun_local']
    if rows:
        print(f'Resuming — already done: {[r["name"] for r in rows]}')

# ── Main loop ────────────────────────────────────────────────────────
for q, info in QUANTS.items():
    row_name = f'llama_cpp_{q}'
    if any(r['name'] == row_name for r in rows):
        print(f'[{q}] already done, skipping')
        continue

    dest = GGUF_DIR / info['file']
    if not dest.exists():
        print(f'Downloading {info["file"]}...')
        hf_hub_download(repo_id=GGUF_REPO, filename=info['file'],
                        local_dir=str(GGUF_DIR))
    print(f"\n{'='*60}\n{q} ({dest.stat().st_size // 1024 // 1024} MB)")

    t0 = time.monotonic()
    ppl, n_ev = compute_ppl(dest)
    wall = time.monotonic() - t0
    print(f'>>> {q}: PPL={ppl:.4f}, n_tokens_eval={n_ev}, {wall:.1f}s')

    rows.append({
        'name':              row_name,
        'description':       f'llama-cpp-python {q}, stride-matched eval.',
        'bpw_reported':      info['bpw'],
        'perplexity':        round(ppl, 4),
        'n_tokens':          n_ev,
        'stride':            STRIDE,
        'context':           CONTEXT,
        'wallclock_seconds': round(wall, 2),
        'model':             'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'model_revision':    None,
        'dataset':           'wikitext/wikitext-2-raw-v1/test',
        'source':            'rerun_local',
        'notes':             'llama-cpp-python + TheBloke GGUF. '
                             'stride=512, context=2048.',
    })
    OUTPUT.write_text(json.dumps(rows, indent=2) + '\n')
    print(f'Saved {len(rows)}/4 rows to {OUTPUT}')

print(f'\nDone. {len(rows)}/4 quants. source=rerun_local, stride={STRIDE}')


In [ ]:
# Cell 7: Phase E — generate the perplexity-vs-bpw plot
import os
os.makedirs("docs", exist_ok=True)
!python -m research.quant.plot_results \
    --inputs research/quant/results/srd_sweep.json,research/quant/results/kquant_sweep.json \
    --output docs/srd_perplexity_vs_bpw.png
from IPython.display import Image
Image("docs/srd_perplexity_vs_bpw.png")

In [ ]:
# Cell 8: Show the verdict + download artifacts
import json
from pathlib import Path

srd = json.load(open("research/quant/results/srd_sweep.json"))
kq  = json.load(open("research/quant/results/kquant_sweep.json"))

print("─── SRD sweep ──────────────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}")
for r in srd:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}")

print("\n─── K-quant baseline ───────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}  source")
for r in kq:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}  {r.get('source', '?')}")

print("\n─── Verdict (pre-committed decision rule) ──────────")
srd_a1 = next((r for r in srd if r['name'].startswith('srd_alpha_1.0_g')), None)
q6 = next((r for r in kq if 'Q6_K' in r['name']), None)
if srd_a1 and q6:
    margin = q6['perplexity'] - srd_a1['perplexity']
    print(f"SRD α=1.0 @ {srd_a1['bpw_reported']:.2f} bpw: PPL = {srd_a1['perplexity']:.3f}")
    print(f"Q6_K     @ {q6['bpw_reported']:.2f} bpw: PPL = {q6['perplexity']:.3f}")
    print(f"Margin (Q6_K - SRD): {margin:+.3f}")
    if margin >= 0.05:
        print("→ VERDICT: SRD beats Q6_K by ≥0.05 PPL. Worth pursuing per the plan.")
    elif margin > 0:
        print("→ VERDICT: SRD edges Q6_K but inside the 0.05 noise margin. Not worth the ~2× memory.")
    else:
        print("→ VERDICT: SRD loses to Q6_K at lower bpw. Shelve.")

# Trigger Colab download — works in Colab only.
try:
    from google.colab import files
    for path in ["research/quant/results/srd_sweep.json",
                 "research/quant/results/kquant_sweep.json",
                 "docs/srd_perplexity_vs_bpw.png"]:
        if Path(path).exists():
            files.download(path)
except ImportError:
    print("\n(google.colab not available — artifacts left at the paths above)")